In [24]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx

In [25]:
sys.path.insert(0, "src")

from grn import grn

np.random.seed(42)


In [26]:
G = grn().add_structure(
    n=200,
    k=1,
    alpha=1e-99,
    beta=0.75,       # 1 - 1/r
    gamma=0.25,      # 1/r
    kappa=10,        # --w
    delta_in=100,
    delta_out=1,
)

In [27]:
G.simulate_rna(
    s=1e-4,
    dt=1e-2,
    tmax=20000,
    burnin=5000,
    tol=1e-3,
    save=True,
)


array([[0.20285071, 0.25095562, 0.3685564 , 0.23052361, 0.09791689,
        0.20806987, 0.05890962, 0.16453678, 0.20238643, 0.09326593,
        0.09646142, 0.64892876, 0.02688249, 0.29364252, 0.0240407 ,
        0.17761111, 0.01538368, 0.18128911, 0.04949782, 0.06859177,
        0.14914242, 0.20321983, 0.45210083, 0.00127327, 0.19928656,
        0.04108859, 0.0342253 , 0.11653443, 0.67624009, 0.1589206 ,
        0.04138443, 0.55217772, 0.12760183, 0.21362012, 0.01788937,
        0.2296222 , 0.62605379, 0.2678963 , 0.56702138, 0.39203113,
        0.48963467, 0.01475103, 0.3418192 , 0.04872582, 0.16750552,
        0.73761168, 0.6945984 , 0.26803118, 0.49270933, 0.57949762,
        0.14091966, 0.09489265, 0.18229322, 0.71652097, 0.70986018,
        0.10837171, 0.16027851, 0.02327928, 0.05158519, 0.18452969,
        0.0970222 , 0.08256984, 0.40724109, 0.18570322, 0.19433295,
        0.6938951 , 0.69570827, 0.01873233, 0.16301367, 0.2126117 ,
        0.24711829, 0.22815284, 0.09991172, 0.35

In [28]:
wt_expression = np.asarray(G.rna).flatten()

wt_df = pd.DataFrame(
    [wt_expression],
    index=["WT"],
    columns=[f"gene_{i}" for i in range(G.n)],
)

wt_df.to_csv("wt_expression.csv")

print("Converged:", G.converged)
print("Expression shape:", wt_expression.shape)
display(wt_df)

Converged: True
Expression shape: (200,)


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_190,gene_191,gene_192,gene_193,gene_194,gene_195,gene_196,gene_197,gene_198,gene_199
WT,0.202851,0.250956,0.368556,0.230524,0.097917,0.20807,0.05891,0.164537,0.202386,0.093266,...,0.133414,0.064493,0.035459,0.439995,0.115955,0.086791,0.486833,0.415345,0.137049,0.294177


In [29]:
ko_expression = np.zeros((G.n, G.n))
ko_logfc = np.zeros((G.n, G.n))

for gene in tqdm(range(G.n), desc="Simulating single-gene KOs"):
    
    result = G.ko_nodes(
        genes=[gene],
        stats=("new_rna", 'logfc'),
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
    )
    
    ko_expression[gene, :] = result["new_rna"].flatten()
    ko_logfc[gene, :] = result["logfc"].flatten()

# Rows = perturbations
# Columns = measured genes
ko_df = pd.DataFrame(
    ko_expression,
    index=[f"KO_{i}" for i in range(G.n)],
    columns=[f"gene_{i}" for i in range(G.n)],
)

fc_ko_df = pd.DataFrame(
    ko_logfc,
    index=[f"KO_{i}" for i in range(G.n)],
    columns=[f"gene_{i}" for i in range(G.n)],
)



# Save KO expression only
ko_df.to_csv("single_KO_expression.csv")

# Also save one convenient combined WT + KO matrix
expression_df = pd.concat([wt_df, ko_df])

expression_df.to_csv("WT_and_single_KO_expression.csv")

print("KO matrix shape:", ko_df.shape)
print("Combined matrix shape:", expression_df.shape)

display(expression_df.head())

Simulating single-gene KOs:   0%|          | 0/200 [00:00<?, ?it/s]

KO matrix shape: (200, 200)
Combined matrix shape: (201, 200)


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_190,gene_191,gene_192,gene_193,gene_194,gene_195,gene_196,gene_197,gene_198,gene_199
WT,0.202851,0.250956,0.368556,0.230524,0.097917,0.208070,0.058910,0.164537,0.202386,0.093266,...,0.133414,0.064493,0.035459,0.439995,0.115955,0.086791,0.486833,0.415345,0.137049,0.294177
KO_0,0.154525,0.391248,0.294031,0.101967,0.129709,0.364894,0.125657,0.135155,0.187353,0.088188,...,0.144786,0.074079,0.019998,0.397664,0.118904,0.089184,0.495365,0.479494,0.137269,0.263566
KO_1,0.348634,0.157430,0.509322,0.370793,0.093092,0.086173,0.020507,0.226762,0.195568,0.099890,...,0.111886,0.049496,0.054959,0.479786,0.112624,0.085264,0.474268,0.372643,0.136760,0.322323
KO_2,0.055702,0.507202,0.233793,0.201084,0.065360,0.627607,0.365830,0.067737,0.151265,0.143018,...,0.174031,0.126078,0.007558,0.375919,0.110802,0.085197,0.533171,0.461349,0.138840,0.272697
KO_3,0.210833,0.245214,0.375153,0.235279,0.094408,0.179416,0.054539,0.167378,0.204260,0.089588,...,0.132358,0.063707,0.039340,0.451029,0.116104,0.086761,0.485944,0.412934,0.137027,0.290562


In [30]:
fc_ko_df.head()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_190,gene_191,gene_192,gene_193,gene_194,gene_195,gene_196,gene_197,gene_198,gene_199
KO_0,-0.392574,0.640650,-0.325917,-1.176817,0.405651,0.810410,1.092913,-0.283795,-0.111355,-0.080766,...,0.118016,0.199919,-0.826340,-0.145937,0.036229,0.039236,0.025067,0.207204,0.002318,-0.158521
KO_1,0.781295,-0.672719,0.466694,0.685700,-0.072905,-1.271761,-1.522410,0.462766,-0.049445,0.098986,...,-0.253872,-0.381821,0.632196,0.124902,-0.042049,-0.025611,-0.037724,-0.156515,-0.003044,0.131821
KO_2,-1.864625,1.015129,-0.656655,-0.197115,-0.583156,1.592794,2.634600,-1.280386,-0.420035,0.616779,...,0.383442,0.967108,-2.230057,-0.227067,-0.065580,-0.026757,0.131173,0.151549,0.018733,-0.109390
KO_3,0.055681,-0.033393,0.025592,0.029460,-0.052648,-0.213762,-0.111212,0.024704,0.013292,-0.058049,...,-0.011464,-0.017700,0.149845,0.035733,0.001848,-0.000510,-0.002637,-0.008399,-0.000231,-0.017839
KO_4,-0.028465,0.024757,-0.017960,-0.162161,0.091011,0.139984,-0.168318,0.014645,-0.187055,-0.188986,...,-0.006777,0.012323,-0.110615,-0.039168,0.006775,0.015527,0.000541,0.004076,0.000228,-0.002336


In [31]:
pairs = np.array(
    list(combinations(range(G.n), 2)),
    dtype=np.int32
)

print("Number of genes:", G.n)
print("Number of double perturbations:", len(pairs))
# For n=200: 19,900

Number of genes: 200
Number of double perturbations: 19900


In [ ]:
n_jobs = 4   # increase if you have enough CPU/RAM

def run_double_ko(pair):
    a, b = pair

    result = G.ko_nodes(
        genes=[a, b],
        stats=("logfc",),
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
    )

    return result["logfc"].flatten()


double_results = Parallel(
    n_jobs=n_jobs,
    pre_dispatch="n_jobs",
    prefer="processes",
    verbose=10,
)(
    delayed(run_double_ko)(pair)
    for pair in pairs
)

double_logfc = np.asarray(double_results, dtype=np.float32)

print("pairs:", pairs.shape)
print("double_logfc:", double_logfc.shape)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    1.1s
[Parallel(n_jobs=4)]: Done   4 tasks      | elapsed:    1.1s
[Parallel(n_jobs=4)]: Done   9 tasks      | elapsed:    1.5s
[Parallel(n_jobs=4)]: Done  14 tasks      | elapsed:    1.7s
[Parallel(n_jobs=4)]: Done  21 tasks      | elapsed:    2.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.19815229536587714s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done  32 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done  50 tasks      | elapsed:    3.4s
[Parallel(n_jobs=4)]: Done  68 tasks      | elapsed:    4.2s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    5.2s
[Parallel(n_jobs=4)]: Done 112 tasks      | elapsed:    6.1s
[Parallel(n_jobs=4)]: Done 138 tasks      | elapsed:    7.2s
[Parallel(n_jobs=4)]: Done 164 tasks      | elapsed:    8.3s
[Parallel(n_jobs=4)]: Done 194 tasks      | elapsed:    9.7s
[Parallel(n_jobs=4)]: Done 224 tasks 

In [ ]:
np.savez_compressed(
    "../small_GRN_all_double_KO.npz",
    pairs=pairs,
    double_logfc=double_logfc,
)

print("Saved small_GRN_all_double_KO.npz")

# Lets work with module 8

In [9]:
import torch
import torch.nn as nn

class PerturbationMLP(nn.Module):
    def __init__(self, n_genes=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_genes, 512),
            nn.GELU(),

            nn.Linear(512, 512),
            nn.GELU(),

            nn.Linear(512, n_genes)
        )

    def forward(self, x):
        return self.net(x)